<a href="https://colab.research.google.com/github/Joaof14/REAL-TIME-EPI/blob/main/notebooks/mask_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Treinamento do Modelo de Detecção de Máscaras

Este documento foi escrito para executar o treinamento do início ao fim no Google Colab. O dataset escolhido contém **853 imagens** e **3 classes** (`with_mask`, `without_mask`, `mask_weared_incorrect`).

## 1. Instalação e Configuração

In [9]:
!pip install -q ultralytics kagglehub opencv-python

## 2. Importação e Carregamento do Dataset


In [10]:
import os
import shutil
import yaml
import xml.etree.ElementTree as ET
from pathlib import Path
from google.colab import files
from tqdm import tqdm


In [11]:
# Importação necessária
import kagglehub
from pathlib import Path
import shutil
import cv2
import xml.etree.ElementTree as ET
from tqdm import tqdm
import random
# Baixa o dataset diretamente para o ambiente do Colab
dataset_path = kagglehub.dataset_download("andrewmvd/face-mask-detection")
print(f"✅ Dataset disponível em: {dataset_path}")

# Copia para um diretório de trabalho com nome amigável
RAW_DATA_DIR = Path('/content/face_mask_raw')
if RAW_DATA_DIR.exists():
    shutil.rmtree(RAW_DATA_DIR)
shutil.copytree(dataset_path, RAW_DATA_DIR)
print(f"✅ Dataset copiado para: {RAW_DATA_DIR}")

Using Colab cache for faster access to the 'face-mask-detection' dataset.
✅ Dataset disponível em: /kaggle/input/face-mask-detection
✅ Dataset copiado para: /content/face_mask_raw


##3. Converter Anotações de PASCAL VOC para YOLO

In [12]:
# Define o diretório de saída para o formato YOLO
YOLO_DATA_DIR = Path('/content/face_mask_yolo')

# Define as classes do dataset (ordem importante!)
CLASS_NAMES = ["with_mask", "without_mask", "mask_weared_incorrect"]

def convert_voc_to_yolo(xml_path, img_width, img_height):
    """
    Converte um único arquivo XML do formato PASCAL VOC para o formato YOLO.
    Retorna uma lista de strings no formato: class_id x_center y_center width height
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()

    yolo_lines = []
    for obj in root.findall("object"):
        class_name = obj.find("name").text
        if class_name not in CLASS_NAMES:
            continue
        class_id = CLASS_NAMES.index(class_name)

        bbox = obj.find("bndbox")
        xmin = int(bbox.find("xmin").text)
        ymin = int(bbox.find("ymin").text)
        xmax = int(bbox.find("xmax").text)
        ymax = int(bbox.find("ymax").text)

        # Converte para formato YOLO (valores normalizados entre 0 e 1)
        x_center = (xmin + xmax) / (2.0 * img_width)
        y_center = (ymin + ymax) / (2.0 * img_height)
        bbox_width = (xmax - xmin) / img_width
        bbox_height = (ymax - ymin) / img_height

        yolo_lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {bbox_width:.6f} {bbox_height:.6f}")

    return yolo_lines

def prepare_yolo_dataset(raw_dir, yolo_dir, train_ratio=0.8):
    """
    Organiza o dataset no formato YOLO, dividindo em treino e validação.
    """
    raw_dir = Path(raw_dir)
    yolo_dir = Path(yolo_dir)

    # Remove diretório YOLO se já existir
    if yolo_dir.exists():
        shutil.rmtree(yolo_dir)

    # Cria estrutura de pastas
    for split in ["train", "val"]:
        (yolo_dir / split / "images").mkdir(parents=True, exist_ok=True)
        (yolo_dir / split / "labels").mkdir(parents=True, exist_ok=True)

    # Lista todas as imagens
    image_extensions = ["*.jpg", "*.jpeg", "*.png"]
    all_images = []
    for ext in image_extensions:
        all_images.extend(list((raw_dir / "images").glob(ext)))

    print(f"📸 Total de imagens encontradas: {len(all_images)}")

    # Embaralha e divide em treino/validação
    import random
    random.seed(42)
    random.shuffle(all_images)

    split_idx = int(len(all_images) * train_ratio)
    train_images = all_images[:split_idx]
    val_images = all_images[split_idx:]

    print(f"📊 Divisão: {len(train_images)} treino / {len(val_images)} validação")

    # Processa cada split
    for split, images in [("train", train_images), ("val", val_images)]:
        print(f"\n🔄 Processando {split}...")

        for img_path in tqdm(images, desc=f"Convertendo {split}"):
            # Copia a imagem
            dest_img_path = yolo_dir / split / "images" / img_path.name
            shutil.copy2(img_path, dest_img_path)

            # Procura o arquivo XML correspondente
            xml_path = raw_dir / "annotations" / (img_path.stem + ".xml")
            if not xml_path.exists():
                continue

            # Obtém as dimensões da imagem
            img = cv2.imread(str(img_path))
            if img is None:
                continue
            img_height, img_width = img.shape[:2]

            # Converte as anotações
            yolo_lines = convert_voc_to_yolo(xml_path, img_width, img_height)

            # Salva o arquivo TXT
            txt_path = yolo_dir / split / "labels" / (img_path.stem + ".txt")
            if yolo_lines:
                with open(txt_path, "w") as f:
                    f.write("\n".join(yolo_lines))

    print("\n✅ Conversão concluída!")

# Executa a conversão do dataset
print("🔄 Iniciando conversão do dataset para o formato YOLO...")
prepare_yolo_dataset(RAW_DATA_DIR, YOLO_DATA_DIR, train_ratio=0.8)

🔄 Iniciando conversão do dataset para o formato YOLO...
📸 Total de imagens encontradas: 853
📊 Divisão: 682 treino / 171 validação

🔄 Processando train...


Convertendo train: 100%|██████████| 682/682 [00:11<00:00, 57.30it/s]



🔄 Processando val...


Convertendo val: 100%|██████████| 171/171 [00:02<00:00, 63.11it/s]


✅ Conversão concluída!


##4. Criação do Arquivo de Configuração (data.yaml)

In [13]:
data_yaml_content = {
    'train': str(YOLO_DATA_DIR / 'train' / 'images'),
    'val': str(YOLO_DATA_DIR / 'val' / 'images'),
    'nc': len(CLASS_NAMES),
    'names': CLASS_NAMES
}

yaml_path = YOLO_DATA_DIR / 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml_content, f, default_flow_style=False)

print(f"✅ Arquivo data.yaml criado em: {yaml_path}")
print("\n📋 Conteúdo do arquivo:")
with open(yaml_path, 'r') as f:
    print(f.read())

✅ Arquivo data.yaml criado em: /content/face_mask_yolo/data.yaml

📋 Conteúdo do arquivo:
names:
- with_mask
- without_mask
- mask_weared_incorrect
nc: 3
train: /content/face_mask_yolo/train/images
val: /content/face_mask_yolo/val/images



##5. Treinamento do Modelo YOLOv8

In [15]:
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [16]:
# Carrega o modelo pré-treinado
model = YOLO('yolov8n.pt')  # Você pode trocar para 'yolov8s.pt' para mais precisão

# Treina o modelo
results = model.train(
    data=str(yaml_path),
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,          # Usa GPU (0 para primeira GPU, 'cpu' para CPU)
    workers=2,
    patience=50,       # Early stopping (interrompe se não melhorar por 50 épocas)
    save=True,
    save_period=10,
    project='/content/runs/detect',
    name='face_mask_detection',
    exist_ok=True
)

print("✅ Treinamento concluído!")

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/face_mask_yolo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=face_mask_detection, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True

##6. Validação e Download do Modelo

In [17]:
# Carrega o melhor modelo salvo durante o treinamento
best_model_path = '/content/runs/detect/face_mask_detection/weights/best.pt'
model = YOLO(best_model_path)

# Valida o modelo no conjunto de validação
metrics = model.val()
print(f"\n📊 Métricas de Validação:")
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.p[0]:.4f}")
print(f"Recall: {metrics.box.r[0]:.4f}")

# Faz o download do arquivo best.pt
from google.colab import files
files.download(best_model_path)

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3423.1±1257.7 MB/s, size: 570.2 KB)
val: Scanning /content/face_mask_yolo/val/labels.cache... 171 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 171/171 59.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 11/11 2.1it/s 5.2s
                   all        171        763      0.844      0.808       0.85      0.604
             with_mask        151        557      0.925      0.953      0.978      0.709
          without_mask         60        190      0.911      0.758      0.876      0.568
 mask_weared_incorrect         14         16      0.695      0.713      0.695      0.535
Speed: 3.6ms preprocess, 8.0ms inference, 0.0ms loss, 2.6ms postprocess per image
Results saved to /content/runs/detect/val

📊 Métr

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>